# Collision Avoidance Classification Model Training (MobileNetV2 + ONNX Export)

This notebook trains a lightweight **MobileNetV2 Binary Classifier** (`free` vs `blocked`) on collision avoidance image datasets (`dataset/free` and `dataset/blocked`).

### Key Features:
1. **MobileNetV2 Lightweight Architecture** (optimized for Jetson Nano / JetRacer).
2. **90% Train / 10% Test Dataset Split** (`torch.utils.data.random_split`).
3. **PyTorch CrossEntropyLoss API**.
4. **Side-by-side Loss & Accuracy Widgets** (`ipywidgets.HBox`).
5. **Live Image Preview Widget with Prediction Status**.
6. **Real-time Dual Matplotlib Charts (Loss & Accuracy Curves)**.
7. **Automatic Model Export**: PyTorch (`best_model_mobilenet_trt.pth`) and ONNX (`Collision Avoidance MobileNet Model`).


### 1. Setup Environment & Load Free/Blocked Dataset


In [7]:
import os
import sys
import glob
import time
import torch
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
import numpy as np
import cv2
from PIL import Image
from pathlib import Path

# Add parent search paths to sys.path
pass # curr = Path.cwd()
for p in [curr, curr.parent, curr.parent.parent]:
    if p.exists() and str(p) not in sys.path:
        pass

try:
    from jetracer_ai.utils import bgr8_to_jpeg
except ImportError:
    from utils import bgr8_to_jpeg

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Using PyTorch Device: {device}")

# Dataset Paths
dataset_dir = os.path.join(Path.cwd(), 'datasets/urban_traffic/classification')
if not os.path.exists(dataset_dir):
    dataset_dir = os.path.join(Path.cwd().parent, 'datasets/urban_traffic/classification')

free_dir = os.path.join(dataset_dir, 'free')
blocked_dir = os.path.join(dataset_dir, 'blocked')

os.makedirs(free_dir, exist_ok=True)
os.makedirs(blocked_dir, exist_ok=True)

# Image Transformations
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

transform = transforms.Compose([
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

# Load ImageFolder Dataset
try:
    def is_valid_image(path):
        try:
            if os.path.getsize(path) == 0:
                return False
            with Image.open(path) as img:
                img.verify()
            return True
        except Exception:
            return False
    dataset = torchvision.datasets.ImageFolder(dataset_dir, transform=transform, is_valid_file=is_valid_image)
    total_samples = len(dataset)
    class_names = dataset.classes
    print(f"[+] Dataset loaded: Total {total_samples} images | Classes: {class_names}")
except Exception as e:
    dataset = None
    total_samples = 0
    class_names = ['blocked', 'free']
    print(f"[*] Dataset notice: {e}")


[+] Using PyTorch Device: cuda
[+] Dataset loaded: Total 617 images | Classes: ['blocked', 'free']


### 2. Initialize MobileNetV2 Model & Fine-Tuning Check


In [8]:
pth_save_path = os.path.join(Path.cwd(), 'models/urban_traffic/best_model_mobilenet_trt.pth')
onnx_save_path = os.path.join(Path.cwd(), 'models/urban_traffic/best_model_mobilenet.onnx')

# MobileNetV2 Binary Classifier (free vs blocked)
try:
    model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
except Exception:
    model = models.mobilenet_v2(pretrained=True)

# Replace classifier head for 2 output classes (blocked, free)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

is_finetuning = False
if os.path.exists(pth_save_path):
    try:
        model.load_state_dict(torch.load(pth_save_path, map_location=device), strict=False)
        is_finetuning = True
        print(f"[+] Existing weights found -> FINE-TUNING mode enabled!")
    except Exception as e:
        print(f"[*] Starting NEW model training ({e}).")
else:
    print("[+] Created NEW MobileNetV2 Classifier model.")

model = model.to(device)


[+] Created NEW MobileNetV2 Classifier model.


### 3. Interactive Training UI & Real-Time Loss/Accuracy Curves


In [9]:
import ipywidgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt

epochs_widget     = ipywidgets.IntText(description='epochs', value=10)
batch_size_widget = ipywidgets.IntText(description='batch size', value=8)

train_loss_widget = ipywidgets.FloatText(description='train loss', layout=ipywidgets.Layout(width='190px'))
test_loss_widget  = ipywidgets.FloatText(description='test loss', layout=ipywidgets.Layout(width='190px'))
train_acc_widget  = ipywidgets.FloatText(description='train acc', layout=ipywidgets.Layout(width='190px'))
test_acc_widget   = ipywidgets.FloatText(description='test acc', layout=ipywidgets.Layout(width='190px'))

progress_widget   = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')
train_button      = ipywidgets.Button(description='Train & Export ONNX', button_style='warning', icon='play')
plot_output       = ipywidgets.Output()

sample_preview_widget = ipywidgets.Image(
    format='jpeg',
    width=224,
    height=224,
    layout=ipywidgets.Layout(border='2px solid #00ff00', border_radius='4px')
)

status_html_widget = ipywidgets.HTML(
    value=f'''
    <div style="font-family: monospace; background: #1e1e1e; color: #00ff00; padding: 10px; border-radius: 6px; width: 320px;">
        <h4 style="margin: 0 0 6px 0; color: #ffffff;">Classification Training</h4>
        <p style="margin: 2px 0;"><b>Mode:</b> {"FINE-TUNING" if is_finetuning else "NEW MODEL"}</p>
        <p style="margin: 2px 0;"><b>Total Samples:</b> {total_samples}</p>
        <p style="margin: 2px 0;"><b>Device:</b> {device}</p>
    </div>
    '''
)

def export_onnx_model():
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    try:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11,
            dynamo=False
        )
    except Exception:
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            verbose=False,
            input_names=['input_0'],
            output_names=['output_0'],
            opset_version=11
        )
    print(f"[+] Successfully exported ONNX Classification model (Opset 11) -> '{onnx_save_path}'")

def start_training(b):
    if dataset is None or total_samples == 0:
        print("[!] ERROR: Dataset is empty! Save 'free' and 'blocked' samples first.")
        return

    epochs = epochs_widget.value
    batch_size = batch_size_widget.value

    # Split dataset 90% Train, 10% Test
    train_size = int(0.95 * len(dataset))
    test_size = len(dataset) - train_size
    if test_size == 0 and len(dataset) > 1:
        test_size = 1
        train_size = len(dataset) - 1

    if train_size > 0 and test_size > 0:
        train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    else:
        train_dataset, test_dataset = dataset, dataset

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
    test_loader  = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    train_button.disabled = True
    start_t = time.time()
    print(f"\n[*] Starting Training (Train: {len(train_dataset)}, Test: {len(test_dataset)}) for {epochs} Epochs...")

    train_loss_history = []
    test_loss_history  = []
    train_acc_history  = []
    test_acc_history   = []

    best_test_acc = -1.0
    best_test_loss = float('inf')

    for epoch in range(epochs):
        # 1. Training Phase
        model.train()
        processed = 0
        sum_train_loss = 0.0
        train_correct = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            count = len(labels)
            processed += count
            sum_train_loss += float(loss) * count
            _, preds = torch.max(outputs, 1)
            train_correct += int(torch.sum(preds == labels.data))

            progress_widget.value = processed / len(train_dataset)
            train_loss_widget.value = sum_train_loss / processed
            train_acc_widget.value = train_correct / processed

            # Live Preview Image with Prediction Banner
            try:
                img_np = images[0].cpu().numpy().transpose(1, 2, 0)
                img_np = (img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])) * 255.0
                img_np = np.clip(img_np, 0, 255).astype(np.uint8)
                img_bgr = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)

                pred_label = class_names[int(preds[0])]
                color = (0, 255, 0) if pred_label == 'free' else (0, 0, 255)
                cv2.putText(img_bgr, f"Pred: {pred_label.upper()}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                sample_preview_widget.value = bgr8_to_jpeg(img_bgr)
            except Exception:
                pass

        # 2. Evaluation Phase
        model.eval()
        sum_test_loss = 0.0
        test_correct = 0
        test_count = 0

        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                
                count = len(labels)
                test_count += count
                sum_test_loss += float(loss) * count
                _, preds = torch.max(outputs, 1)
                test_correct += int(torch.sum(preds == labels.data))

        curr_train_loss = train_loss_widget.value
        curr_test_loss  = sum_test_loss / test_count if test_count > 0 else 0.0
        curr_train_acc  = train_acc_widget.value
        curr_test_acc   = test_correct / test_count if test_count > 0 else 0.0

        test_loss_widget.value = curr_test_loss
        test_acc_widget.value  = curr_test_acc

        train_loss_history.append(curr_train_loss)
        test_loss_history.append(curr_test_loss)
        train_acc_history.append(curr_train_acc)
        test_acc_history.append(curr_test_acc)

        # Best Model Checkpoint Saving
        is_best = False
        if curr_test_acc > best_test_acc or (abs(curr_test_acc - best_test_acc) < 1e-4 and curr_test_loss < best_test_loss):
            best_test_acc  = curr_test_acc
            best_test_loss = curr_test_loss
            is_best = True
            torch.save(model.state_dict(), pth_save_path)
            export_onnx_model()

        best_tag = " ⭐ [BEST MODEL SAVED]" if is_best else ""
        print(f"  Epoch [{epoch+1:02d}/{epochs:02d}] - Train Loss: {curr_train_loss:.4f} (Acc: {curr_train_acc*100:.1f}%) | Test Loss: {curr_test_loss:.4f} (Acc: {curr_test_acc*100:.1f}%){best_tag}")

        # Live Matplotlib Plot
        with plot_output:
            clear_output(wait=True)
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
            ep_range = range(1, len(train_loss_history) + 1)

            # Loss Curve
            ax1.plot(ep_range, train_loss_history, label='Train Loss', color='#1f77b4', marker='o', linewidth=2)
            ax1.plot(ep_range, test_loss_history,  label='Test Loss',  color='#ff7f0e', marker='s', linewidth=2)
            ax1.set_xlabel('Epoch')
            ax1.set_ylabel('Loss')
            ax1.set_title('CrossEntropy Loss Curve')
            ax1.legend()
            ax1.grid(True, linestyle='--', alpha=0.6)

            # Accuracy Curve
            ax2.plot(ep_range, [a*100 for a in train_acc_history], label='Train Acc (%)', color='#2ca02c', marker='o', linewidth=2)
            ax2.plot(ep_range, [a*100 for a in test_acc_history],  label='Test Acc (%)',  color='#d62728', marker='s', linewidth=2)
            ax2.set_xlabel('Epoch')
            ax2.set_ylabel('Accuracy (%)')
            ax2.set_title('Classification Accuracy Curve')
            ax2.legend()
            ax2.grid(True, linestyle='--', alpha=0.6)

            plt.tight_layout()
            plt.show()

    elapsed = time.time() - start_t
    print(f"[+] Training finished in {elapsed:.1f}s!")

    print(f"[+] Training finished in {elapsed:.1f}s! Best Test Acc: {best_test_acc*100:.1f}% | Best Test Loss: {best_test_loss:.4f}")
    print(f"[+] Best Model PyTorch Checkpoint -> '{pth_save_path}'")
    print(f"[+] Best Model ONNX Export -> '{onnx_save_path}'")
    train_button.disabled = False

train_button.on_click(start_training)

train_controls = ipywidgets.VBox([
    epochs_widget,
    batch_size_widget,
    progress_widget,
    ipywidgets.HBox([train_loss_widget, test_loss_widget]),
    ipywidgets.HBox([train_acc_widget, test_acc_widget]),
    train_button
])

train_ui = ipywidgets.VBox([
    ipywidgets.HBox([sample_preview_widget, status_html_widget]),
    train_controls,
    plot_output
])

display(train_ui)
